## PertAdapt on VS Code + a rented GPU box (RunPod / Lambda / Paperspace)

End-to-end pipeline: GCS data access, PertAdapt setup, preprocessing, and training a scFoundation-backed
adapter on the K562 GWPS raw bulk dataset.

**One-time setup per fresh GPU box:**
1. Connect VS Code to the box via the **Remote-SSH** extension (local Macs have no CUDA GPU).
2. `PertAdapt/` is a git submodule of this repo, not something to `git clone` separately - if it's
   ever empty, run `git submodule update --init --recursive`.
3. The scFoundation checkpoint (`PertAdapt/scFoundation/model/models/models.ckpt`, ~1.4GB) is too
   large for git - copy it in separately (e.g. `scp` from wherever you downloaded it) before running
   the training cells near the end.
4. Run `gcloud auth login && gcloud auth application-default login` once per fresh box for GCS access
   (the cell below installs the CLI itself and checks whether this is done).
5. Rental GPU templates (e.g. RunPod's "PyTorch" image) usually ship a torch+CUDA build that already
   matches the driver - the install cell below checks before reinstalling anything.

### 1. GPU check & repo root

In [ ]:
import os
import subprocess

gpu_check = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu_check.returncode != 0:
    print("WARNING: nvidia-smi failed - no GPU visible on this machine. Training will not work here.")
    print(gpu_check.stderr)
else:
    print(gpu_check.stdout)

# Every path below is relative to the repo root rather than a hardcoded Colab-style /content
# directory, so this notebook works the same regardless of where the GPU box checked it out.
REPO_ROOT = subprocess.run(
    ['git', 'rev-parse', '--show-toplevel'], capture_output=True, text=True, check=True
).stdout.strip()
PERTADAPT_ROOT = f"{REPO_ROOT}/PertAdapt"
DATA_DIR = f"{REPO_ROOT}/data"
os.makedirs(DATA_DIR, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("PERTADAPT_ROOT:", PERTADAPT_ROOT)
print("DATA_DIR:", DATA_DIR)

### 2. Install the `gcloud` CLI

In [ ]:
import shutil

# Rented GPU boxes (RunPod/Lambda/Paperspace/etc.) don't ship the gcloud CLI the way a GCE VM
# image does, but it's needed below to read the gs://2026-vcc-llsa bucket.
if shutil.which('gcloud') is None:
    print("gcloud CLI not found - installing...")
    subprocess.run(
        'curl -sSL https://sdk.cloud.google.com | bash -s -- --disable-prompts',
        shell=True, check=True,
    )
    os.environ['PATH'] = os.path.expanduser('~/google-cloud-sdk/bin') + os.pathsep + os.environ['PATH']

subprocess.run(['gcloud', '--version'], check=True)

### 3. PertAdapt code

Already vendored as a git submodule at `PertAdapt/` - no separate clone needed.

In [ ]:
assert os.path.isdir(PERTADAPT_ROOT), (
    f"{PERTADAPT_ROOT} is missing or empty - run `git submodule update --init --recursive` "
    "in a terminal on this box"
)

### 4. Install Python dependencies

In [ ]:
import sys

!{sys.executable} -m pip install -q anndata scanpy dcor local-attention

In [ ]:
# Rental GPU templates (e.g. RunPod's "PyTorch" image, Lambda's stock image) usually already have
# a matching torch+CUDA build preinstalled - check first with:
#   !python -c "import torch; print(torch.__version__, torch.cuda.is_available())"
# If that already prints a version and True, skip this cell entirely to avoid re-downloading
# multi-GB wheels. Otherwise, set this to match the box's actual driver - check with `!nvidia-smi`
# (top right of its output, "CUDA Version: X.Y") - then pick the closest supported tag, e.g. cu121,
# cu124, cu126, cu128.
CUDA_TAG = "cu121"  # <-- EDIT THIS after checking nvidia-smi on your GPU box

# Using `sys.executable -m pip` (not a bare `pip`/`!pip`) guarantees this installs into whatever
# Python this notebook's kernel is actually running, even if a different Python/pip is on PATH.
!{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/{CUDA_TAG}

# Core data science libraries. anndata/scikit-learn kept at environment.yml's pinned versions.
!{sys.executable} -m pip install numpy pandas scipy anndata==0.12.6 scikit-learn==1.5.0
!{sys.executable} -m pip install einops==0.8.2 matplotlib==3.9.1

# Deferred: transformers/datasets aren't used by prep_k562_bulk.py; install later if PertAdapt's
# training scripts turn out to need them.
# !{sys.executable} -m pip install transformers==4.43.3 datasets==2.20.0

In [ ]:
import torch

print("Uninstalling any existing torch_geometric components for a clean, consistent reinstall...")
!{sys.executable} -m pip uninstall -y torch_geometric torch-scatter torch-sparse torch-cluster torch-spline-conv pyg_lib

# Detect the ACTUAL installed torch + CUDA version rather than hardcoding one - a hardcoded string
# breaks the moment torch's version differs from whatever machine this was last run on.
TORCH = torch.__version__.split('+')[0]                                  # e.g. "2.11.0"
CUDA = 'cu' + torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'  # e.g. "cu128"
pyg_url = f"https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html"
print(f"Detected torch {TORCH} / {CUDA} -> using PyG wheel index: {pyg_url}")

!{sys.executable} -m pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f {pyg_url}
!{sys.executable} -m pip install torch_geometric

import torch_geometric
print("torch_geometric installed:", torch_geometric.__version__)

In [ ]:
# PertAdapt's gears/ vendor code imports DataLoader from the pre-2.x torch_geometric location;
# newer torch_geometric moved it to .loader.
!grep -rl "from torch_geometric.data import DataLoader" {PERTADAPT_ROOT}/scFoundation/PertAdapter/gears/ \
  | xargs sed -i 's/from torch_geometric.data import DataLoader/from torch_geometric.loader import DataLoader/'

### 5. Authenticate to Google Cloud

In [ ]:
# Colab's browser popup (google.colab.auth) doesn't exist outside Colab, and unlike a GCE VM, a
# rented GPU box has no built-in GCP service account either - so a fresh box always needs one
# explicit login. This checks whether that's already been done, and falls back to an interactive
# `gcloud auth login` if not (works fine over an SSH terminal in VS Code).
result = subprocess.run(['gcloud', 'auth', 'list', '--format=value(account)'], capture_output=True, text=True)
active_accounts = result.stdout.strip()
if active_accounts:
    print(f"Authenticated as: {active_accounts}")
else:
    print("No active gcloud credentials found - run this in the VS Code terminal (not this cell):")
    print("  gcloud auth login && gcloud auth application-default login")

In [ ]:
!gcloud config set project virtual-cell-challenge-2026

### 6. Browse the training data bucket

In [ ]:
BUCKETtrain = "gs://2026-vcc-llsa/train/"
!gcloud storage ls {BUCKETtrain}*.h5ad

### 7. Download and peek at one training file

In [ ]:
import anndata as ad

specific_file_path = "gs://2026-vcc-llsa/train/K562_gwps_raw_bulk_01.h5ad"
filename = os.path.basename(specific_file_path)
local_path = f"{DATA_DIR}/{filename}"

if not os.path.exists(local_path):
    print(f"Downloading {specific_file_path} -> {local_path}")
    !gcloud storage cp {specific_file_path} {local_path}
else:
    print(f"Already downloaded: {local_path}")

adata_specific = ad.read_h5ad(local_path, backed='r')
print(f"Dimensions: {adata_specific.shape}")
print("obs columns:", adata_specific.obs.columns.tolist())
print(adata_specific.X[:5, :5].toarray() if hasattr(adata_specific.X[:5, :5], "toarray") else adata_specific.X[:5, :5])

### 8. Preprocess the K562 GWPS raw bulk file

In [ ]:
import sys
import numpy as np
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold

# prep_k562_bulk.py: collapses each perturbation to its pseudobulk delta vs. control, holds out
# whole perturbation clusters (not just cells) for a leakage-free ridge-regression sanity check,
# and saves the deltas used to fine-tune PertAdapt.
prep_script_content = """
import sys
import numpy as np
import anndata as ad
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
import os

PATH = sys.argv[1]
MIN_CELLS = 25      # paper's cutoff for calling a perturbation "strong"
TARGET_SUM = 1e4
K = 50              # embedding dimensions
ALPHA = 10.0        # ridge strength; tune this
N_GROUPS, N_FOLDS = 25, 5

a = ad.read_h5ad(PATH)
X = a.X.toarray() if hasattr(a.X, "toarray") else np.asarray(a.X)
X = X.astype(np.float64)
print("shape:", X.shape)
print("obs names:", list(a.obs_names[:3]), "| var columns:", list(a.var.columns), "| var names:", list(a.var_names[:3]))

# 1. Confirm values are mean UMI per cell: value * n_cells should be whole numbers.
for col in ("num_cells_filtered", "num_cells_unfiltered"):
    if col in a.obs:
        n_ = a.obs[col].to_numpy(float)
        print("max |X*{col} - round| = {:.4f}  (~0 => mean UMI per cell over that many cells)".format(np.abs(X * n_[:, None] - np.round(X * n_[:, None])).max(), col=col))
n = a.obs["num_cells_filtered"].to_numpy(float)

# 2. Control rows.
if "core_control" in a.obs:
    ctrl = a.obs["core_control"].astype(str).str.lower().isin(["true", "1", "1.0"]).to_numpy()
elif a.obs_names.str.contains("non-targeting", case=False).any():
    ctrl = a.obs_names.str.contains("non-targeting", case=False).to_numpy()
else:
    ctrl = np.zeros(len(a), bool)
assert ctrl.sum() > 0, "no control rows found; inspect a.obs['core_control'] and a.obs_names"
print("control rows: {} (cells: {})".format(ctrl.sum(), int(n[ctrl].sum())))

# 3. Normalize: rescale each profile to a common total, log1p; control = cell-weighted pooled mean.
def lognorm(M):
    return np.log1p(M / M.sum(axis=1, keepdims=True) * TARGET_SUM)

ctrl_mean = (X[ctrl] * n[ctrl, None]).sum(0) / n[ctrl].sum()
L = lognorm(X)
D = L - lognorm(ctrl_mean[None])[0]          # perturbation effect on every gene

# 4. Map each perturbation to its target gene column ("1_AARS_P1P2_ENSG..." -> AARS).
def target_of(name):
    p = str(name).split("_")
    return p[1] if len(p) >= 4 else str(name)

name_col = next((c for c in ("gene_name", "gene_symbols", "symbol", "gene") if c in a.var), None)
gene_names = a.var[name_col].astype(str).to_numpy() if name_col else a.var_names.to_numpy()
col_of = {g: i for i, g in enumerate(gene_names)}
tgt = np.array([col_of.get(target_of(s), -1) for s in a.obs_names])

keep = (~ctrl) & (n >= MIN_CELLS) & (tgt >= 0)
print("perturbations kept: {} of {} (>= {} cells and target gene measured)".format(keep.sum(), (~ctrl).sum(), MIN_CELLS))
D, tgt, n_keep = D[keep], tgt[keep], n[keep]

# 5. Groups for the split: cluster perturbations by their effect profile, hold out whole clusters.
Z = np.linalg.svd(D - D.mean(0), full_matrices=False)
pcs = Z[0][:, :10] * Z[1][:10]
groups = KMeans(N_GROUPS, n_init=10, random_state=0).fit_predict(pcs)

def pearson_rows(A, B):
    A, B = A - A.mean(1, keepdims=True), B - B.mean(1, keepdims=True)
    return (A * B).sum(1) / np.sqrt((A * A).sum(1) * (B * B).sum(1) + 1e-12)

res = {m: [] for m in ("control (zero)", "train mean", "ridge")}
strength = np.linalg.norm(D, axis=1)
strong = strength >= np.quantile(strength, 0.75)
for tr, te in GroupKFold(N_FOLDS).split(D, groups=groups):
    Dtr = D[tr]
    _, S, Vt = np.linalg.svd(Dtr - Dtr.mean(0), full_matrices=False)
    G = Vt[:K].T * S[:K]                                # gene embeddings from TRAIN rows only
    model = Ridge(alpha=ALPHA).fit(G[tgt[tr]], Dtr)     # perturbation embedding = its target gene's embedding
    preds = {"control (zero)": np.zeros((len(te), D.shape[1])),
             "train mean": np.tile(Dtr.mean(0), (len(te), 1)),
             "ridge": model.predict(G[tgt[te]])}
    for m, P in preds.items():
        res[m].append(np.c_[te, pearson_rows(P, D[te]), ((P - D[te]) ** 2).mean(1)])

print("\nheld-out clusters, {}-fold (median Pearson on delta / mean MSE)".format(N_FOLDS))
print("{:16s} {:>8s} {:9s} {:>9s} {:>11s}".format('model', 'all r', 'strong r', 'all MSE', 'strong MSE'))
for m, r in res.items():
    r = np.vstack(r)
    s = strong[r[:, 0].astype(int)]
    print("{:16s} {:8.3f} {:9.3f} {:9.4f} {:11.4f}".format(m, np.median(r[:, 1]), np.median(r[s, 1]), r[:, 2].mean(), r[s, 2].mean()))

np.savez_compressed("k562_gwps_delta.npz", D=D, target_col=tgt, n_cells=n_keep,
                    genes=gene_names, perts=a.obs_names[keep].to_numpy())
print("saved k562_gwps_delta.npz")
"""

os.chdir(PERTADAPT_ROOT)
with open("prep_k562_bulk.py", "w") as f:
    f.write(prep_script_content)

print(f"Running prep_k562_bulk.py against {local_path}...")
!python prep_k562_bulk.py {local_path}
print(f"Done. k562_gwps_delta.npz written to {PERTADAPT_ROOT}/")

### 9. scFoundation checkpoint

In [ ]:
model_checkpoint_dir = f"{PERTADAPT_ROOT}/scFoundation/model/models"
model_checkpoint_path = f"{model_checkpoint_dir}/models.ckpt"
os.makedirs(model_checkpoint_dir, exist_ok=True)

if os.path.exists(model_checkpoint_path):
    size_gb = os.path.getsize(model_checkpoint_path) / 1e9
    print(f"Checkpoint already present: {model_checkpoint_path} ({size_gb:.2f} GB)")
else:
    # PertAdapt's own download.txt only lists OneDrive/SharePoint folders, which aren't
    # script-downloadable - this HuggingFace mirror is a direct URL instead.
    model_download_url = 'https://huggingface.co/zhaoxuan-li/scFoundation-model/resolve/main/models/models.ckpt'
    print(f"Downloading from {model_download_url}")
    !wget -O {model_checkpoint_path} {model_download_url}

### 10. Train

Adapted from `run_norman.sh` for our K562 bulk data on a single-GPU box:
- `data_dir` / `data_name`: point at the `k562_gwps_delta.npz` produced above.
- `singlecell_model_path`: the checkpoint located in the previous cell.
- `nproc_per_node=1`: a rented GPU box typically has a single GPU.
- `result_dir`: a fresh timestamped directory under `scFoundation/PertAdapter/results/`.

In [ ]:
import datetime

os.chdir(f"{PERTADAPT_ROOT}/scFoundation/PertAdapter/run_sh")

pertadapt_data_dir = f"{PERTADAPT_ROOT}/data/"
os.makedirs(pertadapt_data_dir, exist_ok=True)

# k562_gwps_delta.npz was written directly to PERTADAPT_ROOT by the preprocessing cell above.
current_npz_path = f"{PERTADAPT_ROOT}/k562_gwps_delta.npz"
target_npz_path = os.path.join(pertadapt_data_dir, 'k562_gwps_delta.npz')
if os.path.exists(current_npz_path):
    !mv {current_npz_path} {target_npz_path}
    print(f"Moved k562_gwps_delta.npz -> {target_npz_path}")
elif not os.path.exists(target_npz_path):
    print(f"Warning: {current_npz_path} not found - did the preprocessing cell run successfully?")

gene2go_pkl_path = os.path.join(pertadapt_data_dir, 'gene2go.pkl')
if not os.path.exists(gene2go_pkl_path):
    gene2go_pkl_url = 'https://raw.githubusercontent.com/biomap-research/scFoundation/main/data/pert/gene2go.pkl'
    print(f"Downloading gene2go.pkl -> {gene2go_pkl_path}")
    !wget -O {gene2go_pkl_path} {gene2go_pkl_url}
else:
    print(f"gene2go.pkl already present: {gene2go_pkl_path}")

data_dir = '../../../data/'          # relative to run_sh/, matches pertadapt_data_dir
data_name = 'k562_gwps_delta'
seed = 1
epochs = 20
valid_every = 1
batch_size = 4
accumulation_steps = 2
test_batch_size = 4
hidden_size = 512
bin_set = 'autobin_resolution_append'
model_type = 'maeautobin'
finetune_method = 'frozen'
singlecell_model_path = model_checkpoint_path   # from the checkpoint cell above
mode = 'v1'
highres = 0
lr = 0.01

timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
result_base_dir = f"{PERTADAPT_ROOT}/scFoundation/PertAdapter/results/"
result_dir = os.path.join(result_base_dir, f'pa_adj_weighted_de_loss_k562_bulk-{timestamp}')
os.makedirs(result_dir, exist_ok=True)
print(f"Result directory: {result_dir}")

command = f"""
python -m torch.distributed.run \
    --standalone \
    --nnodes=1 --nproc_per_node 1 \
    ../train_ddp.py \
    --data_dir={data_dir} \
    --data_name={data_name} \
    --seed={seed} \
    --result_dir={result_dir} \
    --epochs={epochs} \
    --valid_every={valid_every} \
    --batch_size={batch_size} \
    --test_batch_size={test_batch_size} \
    --hidden_size={hidden_size} \
    --bin_set={bin_set} \
    --model_type={model_type} \
    --finetune_method={finetune_method} \
    --singlecell_model_path={singlecell_model_path} \
    --mode={mode} \
    --highres={highres} \
    --accumulation_steps={accumulation_steps} \
    --loss='loss_adapt' \
    --ddp_loss_weight=5.0 \
    --lr={lr} \
    --model_class=\"GEARS_Model_Pert_Adapter_New\" \
    --proj_name=\"PertAdapt\" \
    --exp_name=\"pa_adj_weighted_de_loss_k562_bulk-{timestamp}\" > {result_dir}/train.log 2>&1
"""
print(command)
!{command}
print(f"Done. Check {result_dir}/train.log")

### 11. Check the training log

In [ ]:
log_file_path = os.path.join(result_dir, 'train.log')

if os.path.exists(log_file_path):
    print(f"Contents of {log_file_path}:")
    with open(log_file_path, 'r') as f:
        print(f.read())
else:
    print(f"Error: Log file not found at {log_file_path}")

### Next steps

Training writes checkpoints and metrics under `result_dir`. Evaluating or running inference with
the trained adapter isn't wired up in this notebook yet - see PertAdapt's `results_process/`
directory and its README for the expected evaluation workflow.